### Topic 9: PII and Access Scoping

### Why Do We Need Access Scoping?

Suppose a banking application stores customer records in Qdrant.

Each record contains:

- Customer Name
- Mobile Number
- Account Number
- FD Number
- FD Status

Example:

```text
Customer Name : Shobha Chopra
FD Number     : BJ2019FD7717
Status        : Closed (Premature)
Mobile Number : 9876543210
```

This information is called **Personally Identifiable Information (PII)** because it can identify a customer.

---

### What Is The Risk?

Suppose the customer asks:

> What is my FD status?

Without access scoping, Qdrant searches every customer record.

It may return:

- Shobha Chopra
- Ramesh Patel
- Anita Sharma

This exposes another customer's personal information.

This is called **Cross-Customer Data Leakage**.

---

### What Is The Correct Solution?

The solution has two parts:

- Store customer records in a separate collection.
- Filter every search using the authenticated customer's FD Number.

Example:

```text
fd_no = BJ2019FD7717
```

Now Qdrant searches only this customer's record.

Other customer records are ignored.

---

### Why Use Separate Collections?

Suppose all documents are stored together.

```text
Policy Documents
Customer Records
FAQs
```

A policy search may accidentally return customer records.

Instead, create separate collections.

```text
Policy Collection

- FAQs
- Policies
- Product Guides
```

```text
Customer Collection

- Customer Records
```

Now a policy search can never return customer PII because customer records are stored in a different collection.

---

### Right To Be Forgotten

Suppose a customer requests deletion of their data.

Deleting the customer from the database is not enough.

The corresponding vector must also be deleted from Qdrant.

Otherwise, the customer's information still exists inside the vector database.

---

### Code Flow

The program executes in the following order:

1. Load the embedding model.
2. Create two Qdrant collections.
3. Store policy documents in the Policy collection.
4. Store customer records in the Customer collection.
5. Demonstrate the risk of searching without access scoping.
6. Apply caller-scoped filtering using the customer's FD Number.
7. Demonstrate that policy searches cannot return customer records.
8. Delete a customer record from Qdrant.
9. Verify that the deleted customer can no longer be found.

---

### Output Explanation

#### TensorFlow Warning

```text
WARNING:tensorflow...
```

This warning comes from TensorFlow.

It does not affect the Qdrant demonstration and can be ignored.

---

#### Setting Up Collections

```text
Policy collection: 3 points
Customer collection: 3 points
```

The notebook creates two separate collections.

- Policy Collection contains only public documents.
- Customer Collection contains only customer records.

This architectural separation prevents accidental exposure of customer data.

---

#### Unscoped Search

```text
RISK: Unscoped search returns other customers' PII
```

Output:

```text
Shobha Chopra
Ramesh Patel
Anita Sharma
```

The search is performed without any filter.

Qdrant searches every customer record and returns multiple customers.

This demonstrates **Cross-Customer Data Leakage**.

---

#### Caller-Scoped Search

```text
FIX: Caller-scoped search
```

Output:

```text
Returned:

fd_no = BJ2019FD7717
name = Shobha Chopra
status = Closed (Premature)
```

The search applies the filter:

```text
fd_no = BJ2019FD7717
```

Only the authenticated customer's record is returned.

No other customer's data is exposed.

---

#### Policy Search

```text
Policy search returned 3 results
```

The search is executed only against the Policy collection.

Customer records are stored in another collection.

Therefore, customer PII can never appear in the search results.

---

#### Right To Be Forgotten

```text
Before:
point exists = True
```

The customer record exists inside Qdrant.

After deletion:

```text
point exists = False
```

The vector has been removed successfully.

This satisfies the vector database part of the deletion process.

The source database or CSV must also delete the same customer.

---

#### Search After Deletion

```text
No record found for BJ2021FD4432
```

The deleted customer can no longer be retrieved from Qdrant.

This confirms that the deletion was successful.

---

### Real-World Issues, Edge Cases & Debugging

**Unscoped Search**

Searching customer records without caller filtering may expose another customer's PII.

Always filter using the authenticated customer ID.

---

**Mixed Collections**

Storing policy documents and customer records in the same collection increases the risk of accidental PII exposure.

Store them in separate collections.

---

**Incomplete Deletion**

Deleting a customer only from the database leaves a copy inside Qdrant.

Always delete the vector as well.

---

**Filtering Is Not Security**

Metadata filtering limits search results.

It does not prevent someone with direct database access from reading the stored payload.

Infrastructure-level security such as authentication and encryption is also required.

---

### Common Mistakes

- Storing customer records and policy documents in the same collection.
- Forgetting caller-scoped filtering.
- Deleting customer data only from the source database.
- Assuming metadata filtering alone provides complete security.

---

### Lead-Level Interview Questions

**Q1. Why should customer records and policy documents use separate collections?**

Separate collections provide architectural isolation. A policy search can never return customer records.

---

**Q2. What is caller-scoped filtering?**

Caller-scoped filtering limits search results to the authenticated customer's records using metadata such as `fd_no`.

---

**Q3. What is Cross-Customer Data Leakage?**

It occurs when one customer's query returns another customer's personal information.

---

**Q4. Why is metadata filtering not sufficient for security?**

Metadata filtering controls search results, but it does not protect data from users with direct access to the vector database.

---

**Q5. What is Right To Be Forgotten?**

When a customer requests deletion, their data must be removed from both the source database and the vector database.

---

### Key Takeaways

- Customer PII requires strict access control.
- Store customer records and policy documents in separate collections.
- Always apply caller-scoped filtering.
- Delete vectors as well as source records.
- Metadata filtering improves security but is not a replacement for infrastructure-level access control.

---

### Code Flow

The program executes in the following order:

1. Load the embedding model.
2. Create an in-memory Qdrant client.
3. Create two collections:
   - **Policy Collection** for FAQs, policies, and product guides.
   - **Customer Collection** for customer records containing PII.
4. Convert policy documents into embeddings and store them in the Policy Collection.
5. Convert customer records into embeddings and store them in the Customer Collection.
6. Create a payload index on `fd_no` to enable fast caller-scoped filtering.
7. Perform an unscoped search on the Customer Collection to demonstrate PII leakage.
8. Perform a caller-scoped search using the authenticated customer's `fd_no`.
9. Search the Policy Collection to show that customer records cannot be returned because they are stored in a separate collection.
10. Delete a customer's vector from the Customer Collection to demonstrate the **Right To Be Forgotten**.
11. Search again for the deleted customer to verify that the record has been removed.

---

### Output Explanation

#### TensorFlow Warning

```text
WARNING:tensorflow...
```

This is a TensorFlow deprecation warning generated while loading the embedding model.

It does not affect the execution of the notebook and can be ignored.

---

#### Setting Up Collections

```text
Policy collection: 3 points (no PII)
Customer collection: 3 points (full record embedding)
```

The notebook creates two separate collections.

- The **Policy Collection** stores only public documents such as FAQs and policies.
- The **Customer Collection** stores customer records containing PII.

This separation prevents policy searches from accidentally returning customer records.

---

#### Unscoped Search

```text
RISK: Unscoped search returns other customers' PII
```

Output:

```text
Shobha Chopra
Ramesh Patel
Anita Sharma
```

The search is performed without any metadata filter.

Qdrant searches all customer records and returns multiple customers because their embeddings are semantically similar.

This demonstrates **Cross-Customer Data Leakage**.

---

#### Caller-Scoped Search

```text
FIX: Caller-scoped search
```

Output:

```text
fd_no=BJ2019FD7717
name=Shobha Chopra
status=Closed (Premature)
```

The search applies the following filter:

```text
fd_no = BJ2019FD7717
```

Qdrant searches only the authenticated customer's record.

No other customer's information is returned.

---

#### Policy Search

```text
Policy search returned 3 result(s)
```

The search is performed only on the Policy Collection.

Only policy-related documents are returned.

Customer records cannot appear because they are stored in a different collection.

---

#### Right To Be Forgotten

Before deletion:

```text
point exists = True
```

The customer's vector exists in Qdrant.

After deletion:

```text
point exists = False
```

The vector has been successfully removed from the Customer Collection.

The notebook also reminds us that the same customer must be deleted from the source database or CSV.

---

#### Search After Deletion

```text
No record found for BJ2021FD4432
```

The deleted customer can no longer be retrieved.

This confirms that the deletion was successful.

---

#### Payload Index Warning

```text
Payload indexes have no effect in the local Qdrant.
```

This warning appears because the notebook uses:

```python
QdrantClient(":memory:")
```

The in-memory version of Qdrant does not support payload indexes.

When Qdrant runs as a server using Docker or Qdrant Cloud, payload indexes improve filtering performance.

The warning does not affect the demonstration because the notebook contains only a few records.

---

In [1]:
"""
qdrant_pii_scoping.py
-----------------------
PII and access-scoping in practice.

Shows:
  1. The risk: unscoped search leaks another customer's PII
  2. The wrong fix: doc_type filter in a mixed collection (still fragile)
  3. The right fix: separate collection + caller-scoped filtering
  4. Right-to-be-forgotten: deleting a point when a customer requests it
  5. Redaction before embedding: only embed non-PII fields for semantic
     search, keep PII in the structured database only

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    PayloadSchemaType,
    Filter,
    FieldCondition,
    MatchValue,
    PointIdsList,
)
from sentence_transformers import SentenceTransformer

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
VECTOR_SIZE = 384

# two collections -- the key architectural decision
POLICY_COLLECTION = "fd_policy_docs"      # public knowledge, no PII
CUSTOMER_COLLECTION = "fd_customer_records"  # PII, caller-scoped

client = QdrantClient(":memory:")
model = SentenceTransformer(MODEL_NAME)

# customer records -- what fd_master_database.csv rows look like as Documents
CUSTOMER_RECORDS = [
    {
        "fd_no": "BJ2019FD7717",
        "customer_name": "Shobha Chopra",
        "mobile": "9876543210",
        "account_no": "01845146270482",
        "amount_inr": 391000,
        "status": "Closed (Premature)",
        "branch": "Pune",
        "tenure_months": 24,
    },
    {
        "fd_no": "BJ2021FD4432",
        "customer_name": "Ramesh Patel",
        "mobile": "9823456781",
        "account_no": "02934857163920",
        "amount_inr": 250000,
        "status": "Active",
        "branch": "Pune",
        "tenure_months": 24,
    },
    {
        "fd_no": "BJ2022FD8891",
        "customer_name": "Anita Sharma",
        "mobile": "9765432109",
        "account_no": "03847261938471",
        "amount_inr": 500000,
        "status": "Active",
        "branch": "Mumbai",
        "tenure_months": 36,
    },
]

# policy chunks -- no PII
POLICY_CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the penalty for early FD closure? A 1 percent penalty applies.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
]


def make_point_id(key: str) -> int:
    """Deterministic ID from any string key -- same input always gives same ID."""
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


def record_to_text(record: dict) -> str:
    """Serialise a customer record to the same 'key: value' format the CSV
    loader produces -- this is what gets embedded."""
    return "\n".join(f"{k}: {v}" for k, v in record.items())


def record_to_redacted_text(record: dict) -> str:
    """Embed only non-PII fields -- branch, tenure, status, amount tier.
    PII (name, mobile, account_no) stays in the structured DB only."""
    amount_tier = "large" if record["amount_inr"] >= 300000 else "medium" \
        if record["amount_inr"] >= 100000 else "small"
    return (
        f"branch: {record['branch']}\n"
        f"tenure_months: {record['tenure_months']}\n"
        f"status: {record['status']}\n"
        f"amount_tier: {amount_tier}"
    )


def setup_policy_collection() -> None:
    """Policy docs -- public knowledge, no PII, no caller scoping needed."""
    client.create_collection(
        collection_name=POLICY_COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    texts = [c["text"] for c in POLICY_CHUNKS]
    embeddings = model.encode(texts, normalize_embeddings=True)
    points = [
        PointStruct(
            id=make_point_id(POLICY_CHUNKS[i]["source"] + str(i)),
            vector=embeddings[i].tolist(),
            payload={"text": POLICY_CHUNKS[i]["text"],
                     "source": POLICY_CHUNKS[i]["source"],
                     "doc_type": POLICY_CHUNKS[i]["doc_type"]},
        )
        for i in range(len(POLICY_CHUNKS))
    ]
    client.upsert(collection_name=POLICY_COLLECTION, points=points)
    print(f"Policy collection: {len(points)} points (no PII)")


def setup_customer_collection(use_redaction: bool = False) -> None:
    """Customer records -- separate collection, caller-scoped.
    use_redaction=True embeds only non-PII fields; PII is payload-only."""
    client.create_collection(
        collection_name=CUSTOMER_COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )

    # add payload index on fd_no for fast caller-scoped filtering
    client.create_payload_index(
        collection_name=CUSTOMER_COLLECTION,
        field_name="fd_no",
        field_schema=PayloadSchemaType.KEYWORD,
    )

    # choose what to embed -- full record or redacted version
    texts = [
        record_to_redacted_text(r) if use_redaction else record_to_text(r)
        for r in CUSTOMER_RECORDS
    ]
    embeddings = model.encode(texts, normalize_embeddings=True)

    points = [
        PointStruct(
            id=make_point_id(CUSTOMER_RECORDS[i]["fd_no"]),
            vector=embeddings[i].tolist(),
            payload={
                # PII stored in payload -- protected by caller-scoped filter
                "fd_no": CUSTOMER_RECORDS[i]["fd_no"],
                "customer_name": CUSTOMER_RECORDS[i]["customer_name"],
                "mobile": CUSTOMER_RECORDS[i]["mobile"],
                "account_no": CUSTOMER_RECORDS[i]["account_no"],
                "amount_inr": CUSTOMER_RECORDS[i]["amount_inr"],
                "status": CUSTOMER_RECORDS[i]["status"],
                "branch": CUSTOMER_RECORDS[i]["branch"],
                "tenure_months": CUSTOMER_RECORDS[i]["tenure_months"],
            },
        )
        for i in range(len(CUSTOMER_RECORDS))
    ]
    client.upsert(collection_name=CUSTOMER_COLLECTION, points=points)
    label = "redacted embedding" if use_redaction else "full record embedding"
    print(f"Customer collection: {len(points)} points ({label})")


def demo_risk_unscoped_search() -> None:
    """THE RISK: searching customer records with no caller-scoped filter.
    A query about Shobha Chopra's FD also returns Ramesh Patel's record
    because it is semantically similar (same branch, same tenure)."""
    print("\n--- RISK: Unscoped search returns other customers' PII ---")
    query = "FD status for customer in Pune with 24 month tenure"
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # no filter -- searches all customer records
    results = client.query_points(
        collection_name=CUSTOMER_COLLECTION,
        query=query_vector,
        limit=3,
        with_payload=True,
    ).points

    for r in results:
        # returns records for multiple customers -- cross-customer leakage
        print(f"  score={r.score:.4f} | fd_no={r.payload['fd_no']} | "
              f"name={r.payload['customer_name']} | mobile={r.payload['mobile']}")
    print("  All 3 customers' PII returned -- unintended cross-customer leakage")


def demo_caller_scoped_search(authenticated_fd_no: str) -> None:
    """THE FIX: filter by the authenticated caller's fd_no.
    Only the record belonging to this caller is ever a valid result."""
    print(f"\n--- FIX: Caller-scoped search (authenticated as {authenticated_fd_no}) ---")
    query = "what is my FD status"
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # filter by the authenticated caller's fd_no -- set at authentication time
    results = client.query_points(
        collection_name=CUSTOMER_COLLECTION,
        query=query_vector,
        query_filter=Filter(
            must=[FieldCondition(key="fd_no", match=MatchValue(value=authenticated_fd_no))]
        ),
        limit=3,
        with_payload=True,
    ).points

    if results:
        r = results[0]
        print(f"  Returned: fd_no={r.payload['fd_no']} | "
              f"name={r.payload['customer_name']} | status={r.payload['status']}")
        print(f"  Only this customer's record returned -- no cross-customer leakage")
    else:
        print(f"  No record found for {authenticated_fd_no}")


def demo_policy_search_isolation() -> None:
    """Separate collections prevent customer PII from appearing in policy searches.
    A policy search against POLICY_COLLECTION can never return customer records
    because they are not in that collection -- architectural guarantee."""
    print("\n--- SEPARATION: Policy search cannot return customer PII ---")
    query = "premature withdrawal penalty Pune"
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=POLICY_COLLECTION,  # searches ONLY policy docs
        query=query_vector,
        limit=3,
        with_payload=True,
    ).points

    print(f"  Policy search returned {len(results)} result(s):")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")
    print("  No customer PII in these results -- separate collection enforces this")


def demo_right_to_be_forgotten(fd_no: str) -> None:
    """GDPR right-to-be-forgotten: delete the point from Qdrant when a
    customer requests deletion. Deterministic ID means we know exactly
    which point to delete without scanning the collection."""
    print(f"\n--- RIGHT-TO-BE-FORGOTTEN: Delete {fd_no} from vector store ---")

    # confirm the point exists before deletion
    point_id = make_point_id(fd_no)
    before = client.retrieve(
        collection_name=CUSTOMER_COLLECTION,
        ids=[point_id],
        with_payload=True,
        with_vectors=False,
    )
    print(f"  Before: point exists = {len(before) > 0}")
    if before:
        print(f"  PII stored: name={before[0].payload['customer_name']}, "
              f"mobile={before[0].payload['mobile']}")

    # delete the point -- this must also happen when the source CSV row is deleted
    client.delete(
        collection_name=CUSTOMER_COLLECTION,
        points_selector=PointIdsList(points=[point_id]),
    )

    # confirm deletion
    after = client.retrieve(
        collection_name=CUSTOMER_COLLECTION,
        ids=[point_id],
        with_payload=True,
        with_vectors=False,
    )
    print(f"  After:  point exists = {len(after) > 0}")
    print(f"  Point deleted from Qdrant -- must also delete from source CSV/DB")
    total = client.get_collection(CUSTOMER_COLLECTION).points_count
    print(f"  Collection now has {total} points (was {total + 1})")


# ======================================================================
# Run all demos in order
# ======================================================================

print("Setting up collections...")
setup_policy_collection()
setup_customer_collection(use_redaction=False)

# 1. show the risk of unscoped search
demo_risk_unscoped_search()

# 2. show caller-scoped search (the fix)
demo_caller_scoped_search(authenticated_fd_no="BJ2019FD7717")

# 3. show that policy search cannot return customer PII (separate collections)
demo_policy_search_isolation()

# 4. right-to-be-forgotten
demo_right_to_be_forgotten(fd_no="BJ2021FD4432")

# 5. show that after deletion, a scoped search for that fd_no returns nothing
print("\n--- After deletion: caller-scoped search for deleted record ---")
demo_caller_scoped_search(authenticated_fd_no="BJ2021FD4432")


Setting up collections...
Policy collection: 3 points (no PII)
Customer collection: 3 points (full record embedding)

--- RISK: Unscoped search returns other customers' PII ---
  score=0.6793 | fd_no=BJ2021FD4432 | name=Ramesh Patel | mobile=9823456781
  score=0.6760 | fd_no=BJ2019FD7717 | name=Shobha Chopra | mobile=9876543210
  score=0.5544 | fd_no=BJ2022FD8891 | name=Anita Sharma | mobile=9765432109
  All 3 customers' PII returned -- unintended cross-customer leakage

--- FIX: Caller-scoped search (authenticated as BJ2019FD7717) ---
  Returned: fd_no=BJ2019FD7717 | name=Shobha Chopra | status=Closed (Premature)
  Only this customer's record returned -- no cross-customer leakage

--- SEPARATION: Policy search cannot return customer PII ---
  Policy search returned 3 result(s):
  score=0.6699 | [policy] Premature withdrawal incurs a 1 percent penalty on the appli
  score=0.5702 | [faq] What is the penalty for early FD closure? A 1 percent penalt
  score=0.2834 | [product] Senior citi

C:\Users\pauls\AppData\Local\Temp\ipykernel_22256\3759776254.py:142: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(
